# RSA公钥密码算法原理与实现

**摘要：** 详解RSA算法的数学原理、密钥生成、加解密及数字签名实现

- **类别：** 现代密码学
- **难度：** 中等
- **预计用时：** 4小时
- **先修：** 数论基础、模运算、素数概念、哈希函数基础
- **学习目标：** 掌握RSA核心原理与实现

> 说明：本课程强调“可运行 + 可解释 + 可练习”。代码尽量仅使用 Python 标准库（Pyodide 友好）。

## 你将获得什么

- **非对称加密  Asymmetric：** 公钥加密私钥解密
- **数字签名  Signature：** 私钥签名公钥验证
- **因数分解  Factorization：** 

## 学习路线图（从直觉到实现）

我们把学习过程拆成 4 层，每一层都尽量给出“可验证”的产物：

1. **直觉层**：能说清楚它解决什么安全目标，以及为什么需要“密钥/参数/随机性”。
2. **符号层**：能把关键变换写成一个简短公式，例如 $y=f(x,k)$ 或 $$y=f(x,k)\bmod n$$。
3. **实现层**：能写出可运行的函数/类，并通过至少 3 条 `assert` 自测。
4. **对抗层**：能指出它可能被怎么攻（至少一个思路），并用代码做一个最小实验验证。

> 你会发现：能通过“对抗层”的小实验，往往才算真正理解。


## 应用场景与安全性讨论（扩充阅读）

这一主题通常会在以下场景出现（不同主题侧重点不同）：

- **教学/入门**：用简化模型理解“密钥 + 变换”的思想。
- **工程系统**：用成熟算法与标准协议实现保密性/完整性/认证。
- **攻防分析**：通过已知攻击理解“为什么某些方案不再推荐”。

### 安全目标（Security Goals）

常见目标包括：

- **保密性（Confidentiality）**：未授权者无法获得明文信息。
- **完整性（Integrity）**：消息在传输中被篡改能被发现。
- **认证（Authentication）**：确认对方是谁（或确认数据来自谁）。
- **不可否认（Non-repudiation）**：事后不能否认曾经生成/发送过某信息（通常与签名相关）。

### 威胁模型（Threat Model）

做任何“安全性结论”之前，先明确攻击者能做什么：

- 只能看到密文？还是还能选择明文（chosen-plaintext）？
- 能否篡改、重放、插入消息？
- 能否获取部分密钥/随机数？是否存在侧信道？

> 调试提示：如果你觉得“算法没问题但结果不对”，先检查你实现的威胁模型是否一致——例如你是否在复用 nonce/IV，或者把编码/填充当成了算法的一部分。


## 环境准备

In [ ]:
# 课程通用导入（尽量只用标准库）
import math
import random
import string
import secrets
import hashlib
import hmac
import itertools
import statistics
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Iterable

random.seed(0)  # 为了让演示更可复现


## 热身：数据与表示

很多实现问题来自“数据表示”而不是算法本身。请确保你能区分：

- 文本：`str`
- 字节：`bytes`
- 整数：`int`

并能相互转换。

In [ ]:
msg = "Hello, 密码学"
b = msg.encode("utf-8")
print(type(msg), type(b))  # 预期输出：<class 'str'> <class 'bytes'>
print(b[:10])              # 预期输出：前10个字节（与编码有关）
print(b.hex()[:20])        # 预期输出：十六进制字符串前20个字符


### 工具函数：字节/整数/十六进制

In [ ]:
def bytes_to_hex(b: bytes) -> str:
    return b.hex()

def hex_to_bytes(h: str) -> bytes:
    h = h.strip().lower()
    if h.startswith("0x"):
        h = h[2:]
    return bytes.fromhex(h)

def int_to_bytes(x: int, length: int, byteorder: str = "big") -> bytes:
    return x.to_bytes(length, byteorder=byteorder, signed=False)

def bytes_to_int(b: bytes, byteorder: str = "big") -> int:
    return int.from_bytes(b, byteorder=byteorder, signed=False)

assert hex_to_bytes(bytes_to_hex(b"ABC")) == b"ABC"


## 核心内容分步讲解

### Step 1: 了解RSA算法起源与意义

RSA由MIT学者Rivest、Shamir和Adleman于1977年提出，取三人姓氏首字母命名，是第一个实用的公钥密码体制。
在RSA出现前，密码学以对称加密为主，核心痛点是密钥分发难题——通信双方需提前安全共享密钥，无法适应开放网络环境。
1976年Diffie和Hellman提出公钥密码学概念，但未实现完整加密与签名；RSA的突破在于基于大整数因数分解的单向函数特性，首次实现了加密、解密、签名、验证全功能，为互联网安全奠定基础。
RSA于1977年公开发表，1978年申请专利，2000年专利过期后进入公共领域，至今仍是HTTPS、SSH、PGP等协议的核心组件，密钥长度从早期512位演进至当前主流的2048位、4096位。


> 提示：RSA是密码学史上的里程碑，解决了对称加密的密钥分发难题

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 2: 掌握RSA核心数论基础

RSA的数学基石是数论中的模运算、欧拉函数、欧拉定理、模逆元和中国剩余定理，核心知识点如下：
1. 模运算：若$n \mid (a - b)$，则$a \equiv b \pmod{n}$，是RSA加解密的基础运算；
2. 欧拉函数：若$n = pq$（$p、q$为质数），则$\phi(n) = (p-1)(q-1)$，表示1~n中与n互质的整数个数；
3. 欧拉定理：若$\gcd(a, n) = 1$，则$a^{\phi(n)} \equiv 1 \pmod{n}$，是RSA可逆性的核心保障；
4. 费马小定理：欧拉定理的特例，若$p$为素数且$\gcd(a, p) = 1$，则$a^{p-1} \equiv 1 \pmod{p}$；
5. 模逆元：若$ed \equiv 1 \pmod{\phi(n)}$，则$d$是$e$在模$\phi(n)$下的逆元，是私钥生成的关键；
6. 中国剩余定理：用于优化RSA解密过程，将大整数运算拆解为小整数运算，提升解密效率。


> 提示：无需深入推导数论定理，重点掌握应用场景和公式含义

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 3: 理解RSA密钥生成流程

RSA密钥生成是算法的核心步骤，基于大素数和数论特性生成公钥和私钥，具体流程如下：
1. 选取两个大质数$p$和$q$：需保证$p \neq q$，且位数足够（如1024位密钥对应两个512位质数），质数质量直接决定RSA安全性；
2. 计算模数$n$：$n = p \times q$，$n$是公钥和私钥的共同组成部分，也是加解密的核心参数，其位数即密钥长度；
3. 计算欧拉函数$\phi(n)$：根据欧拉函数性质，$\phi(n) = (p-1)(q-1)$，用于后续私钥指数计算；
4. 选择公钥指数$e$：满足$1 < e < \phi(n)$且$\gcd(e, \phi(n)) = 1$（即$e$与$\phi(n)$互质），常用值为65537（既安全又高效，兼顾计算速度和抗攻击能力）；
5. 计算私钥指数$d$：$d$是$e$在模$\phi(n)$下的逆元，即满足$ed \equiv 1 \pmod{\phi(n)}$，通过扩展欧几里得算法求解；
6. 确定密钥对：公钥为$(e, n)$（可公开），私钥为$(d, n)$（需严格保密），同时销毁中间参数$p、q、\phi(n)$。
示例：取$p=61$、$q=53$，则$n=61×53=3233$，$\phi(n)=60×52=3120$，选$e=17$（与3120互质），通过扩展欧几里得算法求得$d=2753$，最终公钥$(17, 3233)$，私钥$(2753, 3233)$。


> 提示：公钥指数$e=65537$是工业标准，避免使用过小的$e$（如3、17）以降低安全风险

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 4: 掌握RSA加解密原理与公式

RSA加解密基于模幂运算，利用公钥加密、私钥解密，且过程可逆，核心公式和流程如下：
### 加密过程
1. 明文预处理：将明文转换为整数$M$，需满足$0 < M < n$（若明文长度超过$n$的比特数，需分段加密）；
2. 模幂运算：使用公钥$(e, n)$计算密文$C$，公式为$C \equiv M^e \pmod{n}$；
示例：明文$M=65$，公钥$(17, 3233)$，则$C=65^{17} \pmod{3233}=2790$。

### 解密过程
1. 密文输入：获取密文$C$，确保$0 < C < n$；
2. 模幂运算：使用私钥$(d, n)$计算明文$M$，公式为$M \equiv C^d \pmod{n}$；
示例：密文$C=2790$，私钥$(2753, 3233)$，则$M=2790^{2753} \pmod{3233}=65$。

### 正确性证明
由私钥生成过程可知$ed \equiv 1 \pmod{\phi(n)}$，即存在整数$k$使得$ed=1+k\phi(n)$。
根据欧拉定理，若$\gcd(M, n)=1$，则$M^{\phi(n)} \equiv 1 \pmod{n}$，因此：
$M^{ed}=M^{1+k\phi(n)}=M×(M^{\phi(n)})^k \equiv M×1^k=M \pmod{n}$，即解密可还原明文。
若$\gcd(M, n)≠1$，可通过中国剩余定理证明仍满足可逆性。


> 提示：加解密均使用模幂运算，实际实现需用“平方-乘”算法优化计算效率

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 5: 理解RSA数字签名原理

RSA不仅可用于加密，还可实现数字签名，核心思想是“私钥签名、公钥验证”，确保消息完整性和不可否认性，流程如下：
### 签名过程
1. 消息哈希：对原始消息$M$计算哈希值$H(M)$（如SHA-256），将哈希值转换为整数$h$，满足$h < n$；
2. 签名生成：使用私钥$(d, n)$对哈希值进行签名，公式为$S \equiv h^d \pmod{n}$，$S$即为签名结果；
### 验证过程
1. 消息哈希：接收方对原始消息$M$重新计算哈希值$H'(M)$，转换为整数$h'$；
2. 签名验证：使用公钥$(e, n)$对签名$S$解密，公式为$h'' \equiv S^e \pmod{n}$；
3. 结果比对：若$h'' = h'$，则签名有效（消息未篡改且来自私钥持有者），否则签名无效。
示例：消息$M=$“This is a signed message”，哈希后$h$，私钥签名得$S$，接收方验证时重新计算哈希$h'$，解密$S$得$h''$，若$h''=h'$则验证通过。
数字签名的核心价值：防止消息被篡改（篡改后哈希值变化）、防止发送方否认（仅私钥持有者能生成有效签名）。


> 提示：签名前必须对消息哈希，避免直接对长消息签名导致效率低下

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 6: 掌握RSA工程实现核心模块

RSA工程实现需解决素数生成、模逆元计算、模幂运算、填充机制等关键问题，核心模块如下：
### 1. 基础工具函数
- 欧几里得算法（gcd）：计算两个数的最大公约数，用于判断$e$与$\phi(n)$是否互质；
- 扩展欧几里得算法（extended_gcd）：求解模逆元，核心用于计算私钥指数$d$；
- Miller-Rabin素性测试（is_prime）：高效判断大整数是否为质数，用于生成$p$和$q$；
- 模幂运算（pow）：优化大整数模幂计算，避免直接计算导致的溢出和低效；
### 2. 填充机制
裸RSA存在安全漏洞（如选择明文攻击、语义安全缺失），需通过填充机制修复，常用PKCS#1 v1.5填充规则：
填充格式：$0x00 || 0x02 || PS || 0x00 || M$，其中PS是至少8字节的随机非零字节，$M$为原始明文；
作用：保证相同明文生成不同密文，限制明文长度，防止数学攻击；
### 3. 字节与整数转换
RSA加解密操作对象是整数，需实现字节数据与整数的相互转换：
- bytes_to_int：将字节数据转换为大整数（大端序）；
- int_to_bytes：将大整数转换为指定长度的字节数据（大端序）。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
def gcd(a, b):
    """计算最大公约数（欧几里得算法）"""
    while b:
        a, b = b, a % b
    return a

def extended_gcd(a, b):
    """扩展欧几里得算法，用于求模逆元"""
    if a == 0:
        return b, 0, 1
    gcd_val, x1, y1 = extended_gcd(b % a, a)
    x = y1 - (b // a) * x1
    y = x1
    return gcd_val, x, y

def mod_inverse(a, m):
    """计算模逆元"""
    gcd_val, x, y = extended_gcd(a, m)
    if gcd_val != 1:
        raise ValueError("模逆元不存在")
    return (x % m + m) % m

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 7: 实现完整RSA加解密与签名功能

基于核心模块实现完整的RSA类，支持密钥生成、字节数据加解密、数字签名与验证，具体实现如下：
1. 类初始化：生成指定长度的密钥对，存储公钥$(n, e)$和私钥$(n, d)$，计算密钥字节长度；
2. 加密方法（encrypt）：接收字符串或字节明文，自动编码、PKCS#1填充、转换为整数、模幂运算，返回字节密文；
3. 解密方法（decrypt）：接收字节密文，转换为整数、模幂运算、转换为字节、PKCS#1去填充、解码，返回明文；
4. 签名方法（sign）：接收字符串或字节消息，计算哈希值、模幂运算（私钥），返回签名整数；
5. 验证方法（verify）：接收消息和签名，计算哈希值、模幂运算（公钥），比对结果返回验证状态；
核心逻辑：封装底层数学运算和数据转换，提供简洁的API接口，屏蔽复杂的实现细节。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
class RSA:
    """RSA加密类"""
    
    def __init__(self, bits=1024):
        """初始化RSA对象，生成密钥对"""
        self.public_key, self.private_key = generate_keypair(bits)
        self.n, self.e = self.public_key
        self.d = self.private_key[1]
        self.key_size = (self.n.bit_length() + 7) // 8
    
    def encrypt(self, message):
        """加密消息：支持字符串或字节"""
        if isinstance(message, str):
            message = message.encode('utf-8')
        return rsa_encrypt_bytes(message, self.public_key)
    
    def decrypt(self, ciphertext):
        """解密消息：返回字节数据"""
        return rsa_decrypt_bytes(ciphertext, self.private_key)
    
    def sign(self, message):
        """数字签名：返回签名整数"""
        if isinstance(message, str):
            message = message.encode('utf-8')
        message_hash = hash(message) % self.n
        return rsa_sign(message_hash, self.private_key)
    
    def verify(self, message, signature):
        """验证签名：返回布尔值"""
        if isinstance(message, str):
            message = message.encode('utf-8')
        message_hash = hash(message) % self.n
        return rsa_verify(message_hash, signature, self.public_key)

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 8: 测试RSA功能正确性与安全性

通过多场景测试验证RSA实现的正确性、安全性和性能，具体测试用例如下：
### 1. 基础加解密测试
- 测试数据：中英文混合字符串（如"Hello, RSA!"、"河南大学的密码学系是全中国最棒的。"）；
- 测试流程：生成512位密钥对→加密明文→解密密文→比对明文与解密结果；
- 验证标准：解密结果与原始明文完全一致，密文为随机十六进制字符串；
### 2. 数字签名与篡改检测测试
- 签名测试：对消息"This is a signed message"生成签名，验证签名有效性；
- 篡改检测：修改消息为"This is a tampered message"，使用原签名验证，确认验证失败；
- 验证标准：合法消息签名验证通过，篡改消息验证失败；
### 3. 性能测试
- 测试内容：1024位密钥生成时间、加密时间、解密时间；
- 参考指标：密钥生成时间约0.05-0.1秒，加密时间微秒级，解密时间毫秒级；
### 4. 安全性演示：大整数分解困难性
- 测试内容：对比小整数（如15=3×5）和中等整数（如1073=29×37）的分解时间；
- 核心结论：小整数可瞬间分解，而RSA使用的大整数（如2048位）分解需天文数字级算力，验证RSA安全性基础。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# 测试加解密
rsa = RSA(512)
test_messages = ["Hello, RSA!", "RSA is a public-key cryptosystem", "河南大学的密码学系是全中国最棒的。"]
for msg in test_messages:
    ciphertext = rsa.encrypt(msg)
    decrypted = rsa.decrypt(ciphertext).decode('utf-8')
    print(f"明文：{msg}")
    print(f"密文：{ciphertext.hex()[:40]}...")
    print(f"解密结果：{decrypted}")
    print(f"验证：{'✓' if msg == decrypted else '✗'}\n")

# 测试数字签名
message = "This is a signed message"
signature = rsa.sign(message)
is_valid = rsa.verify(message, signature)
print(f"签名验证：{'✓ 有效' if is_valid else '✗ 无效'}")

# 测试篡改检测
tampered_msg = "This is a tampered message"
is_tampered_valid = rsa.verify(tampered_msg, signature)
print(f"篡改检测：{'✗ 检测到篡改' if not is_tampered_valid else '✓ 未检测到篡改'}")

# 预期输出：根据输入得到对应结果（请与讲解示例对照）

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 9: 掌握RSA工程实践安全规范

RSA的安全性不仅依赖算法本身，还需严格遵守工程实践规范，避免安全漏洞，核心规范如下：
1. 密钥长度选择：生产环境需使用2048位及以上密钥，1024位密钥已逐渐被淘汰，4096位适用于高安全场景；
2. 密钥管理：私钥需加密存储（如AES加密），避免明文存储或硬编码，定期轮换密钥；
3. 填充机制：禁止使用裸RSA，优先使用RSA-OAEP填充（比PKCS#1 v1.5更安全），避免填充 oracle 攻击；
4. 签名哈希：使用安全哈希函数（如SHA-256、SHA-512），禁止使用弱哈希函数（如MD5、SHA-1）；
5. 性能优化：RSA私钥操作（解密、签名）较慢，避免用于大文件加密，仅用于小数据（如对称密钥、哈希值）；
6. 抗攻击措施：避免重复使用大质数$p、q$生成多个密钥，防止因式分解攻击；禁止使用可预测的质数；
7. 协议适配：在TLS/SSL等协议中使用时，遵循协议规范，避免自定义密钥交换流程；
8. 量子安全：RSA在量子计算面前存在安全风险，高安全场景可考虑后量子密码算法。


> 提示：工程实践中优先使用成熟密码库（如OpenSSL、cryptography），避免自行实现底层算法

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

## 综合示例：端到端流程 + 自测

In [ ]:
# 端到端模板：将主题核心操作封装成接口
# 你可以把 encrypt/decrypt 或 hash/sign/verify 等组合在一起

def pipeline(data: bytes) -> bytes:
    # TODO: 替换为你的端到端流程
    return hashlib.sha256(data).digest()

def check_pipeline():
    a = pipeline(b"hello")
    b = pipeline(b"hello")
    c = pipeline(b"hellp")
    assert a == b
    assert a != c
    print("端到端自测通过")  # 预期输出：端到端自测通过

check_pipeline()


## 小实验：敏感性（雪崩效应）

In [ ]:
def hamming_distance_bytes(a: bytes, b: bytes) -> int:
    if len(a) != len(b):
        raise ValueError("长度必须相同才能计算差异度")
    return sum(x != y for x, y in zip(a, b))

def flip_one_bit(data: bytes, bit_index: int = 0) -> bytes:
    if not data:
        return data
    byte_i = bit_index // 8
    bit_i = bit_index % 8
    byte_i = byte_i % len(data)
    mask = 1 << bit_i
    arr = bytearray(data)
    arr[byte_i] ^= mask
    return bytes(arr)

def core_transform(data: bytes) -> bytes:
    # TODO: 替换成你的核心变换
    # 示例：用 SHA-256 作为“对照组”
    return hashlib.sha256(data).digest()

base = b"hello world"
out1 = core_transform(base)
out2 = core_transform(flip_one_bit(base, 0))

print("输出长度(字节):", len(out1))  # 预期输出：32（若使用SHA-256）
print("差异度(字节):", hamming_distance_bytes(out1, out2))  # 预期输出：通常 > 10


## 附录A：最常用的数学与位运算背景

### 1. 模运算（Modular Arithmetic）

当我们写 $$a \equiv b \pmod n$$ 时，意思是 $a$ 与 $b$ 除以 $n$ 的余数相同，也就是 $n$ 整除 $a-b$。

常见等价写法：

- $a \bmod n$：把 $a$ 映射到 $0..n-1$ 的代表元
- 若 $a<0$，Python 仍保证 $a \bmod n \in [0,n-1]$

**一个很重要的直觉：** 模运算会“折叠”整数轴，把无限多的整数映射到有限集合，所以很多密码体制会用到它来构造“封闭”的运算空间。

### 2. 最大公因数与互质

若 $$\gcd(a,n)=1$$，我们称 $a$ 与 $n$ 互质（coprime）。互质非常重要，因为它意味着 $a$ 在模 $n$ 意义下存在乘法逆元 $a^{-1}$，满足：

$$a\cdot a^{-1} \equiv 1 \pmod n$$

### 3. 扩展欧几里得算法（Extended Euclid）

它不仅能算 $\gcd(a,b)$，还能找到整数 $x,y$ 使得：

$$ax+by=\gcd(a,b)$$

当 $\gcd(a,n)=1$ 时，$x$ 就是 $a$ 关于模 $n$ 的逆元（可能为负，需要再取模）。

### 4. 异或（XOR）

在字节层面，异或常写成 $c=a\oplus b$，它有几个极其重要的性质：

- $a\oplus a=0$
- $a\oplus 0=a$
- $(a\oplus b)\oplus b=a$（可逆性）

因此很多流密码/分组模式会用异或把“密钥流”与明文混合。

> 调试提示：如果你发现解密结果不等于明文，先检查“同一份密钥流是否被一致地使用”，以及字节对齐是否正确。


In [ ]:
# 附录A配套代码：gcd、逆元、异或操作的最小实现与自测

def egcd(a: int, b: int) -> Tuple[int, int, int]:
    """返回 (g, x, y) 使得 a*x + b*y = g = gcd(a,b)"""
    if b == 0:
        return (a, 1, 0)
    g, x1, y1 = egcd(b, a % b)
    x = y1
    y = x1 - (a // b) * y1
    return (g, x, y)

def modinv(a: int, n: int) -> int:
    g, x, _ = egcd(a, n)
    if g != 1:
        raise ValueError("不存在逆元：a 与 n 不互质")
    return x % n

print(math.gcd(3, 26))     # 预期输出：1
print(modinv(3, 26))       # 预期输出：9，因为 3*9=27≡1(mod26)

def xor_bytes(a: bytes, b: bytes) -> bytes:
    if len(a) != len(b):
        raise ValueError("xor 需要等长字节串")
    return bytes(x ^ y for x, y in zip(a, b))

p = b"ABC"
k = b"\x01\x01\x01"
c = xor_bytes(p, k)
p2 = xor_bytes(c, k)
print(c)   # 预期输出：b'@BA'（因为 0x41^1=0x40 等）
print(p2)  # 预期输出：b'ABC'


## 常见错误 / 踩坑点 / 调试提示

> 1. **编码问题**：`str`/`bytes` 混用导致哈希/加解密结果不一致。
>
> 2. **参数合法性**：例如需要互质、需要固定长度、需要随机数不可复用。
>
> 3. **端序与位序**：大端/小端混淆、位操作方向写反。
>
> 4. **测试不足**：缺少边界与异常路径测试。
>
> 5. **把演示当安全方案**：课程中的简化实现不等价于生产安全实现。

## 练习题（含更详细要求）

### 练习1：功能完善（必做）
- 目标：把你的核心函数做成“更好用”的版本  
- 要求：
  - 输入支持 `str` 与 `bytes`
  - 对非法参数给出清晰报错（`ValueError`）
  - 至少写 5 条 `assert`（覆盖正常/边界/异常）

### 练习2：最小攻防实验（推荐）
- 目标：实现一个最小的攻击/对抗脚本  
- 示例方向（任选其一）：
  - 暴力枚举 key space
  - 频率分析/统计偏差
  - 重放/篡改检测失败示例
  - 碰撞/相同摘要的构造（仅演示，不鼓励用于真实攻击）

### 练习3：写一段“工程化清单”（可选）
- 目标：把课程知识迁移到真实系统  
- 建议包含：
  - 参数选择（key 长度、随机数、模式）
  - 依赖库与实现来源（优先标准/权威实现）
  - 安全审计点（日志、异常、边界）


In [ ]:
# 练习参考答案框架（示例）
# 说明：这里只提供“结构”，你需要填入你自己的实现。

def solve_ex1(data):
    if isinstance(data, str):
        data_b = data.encode("utf-8")
    elif isinstance(data, (bytes, bytearray)):
        data_b = bytes(data)
    else:
        raise ValueError("只支持 str/bytes")
    return hashlib.sha256(data_b).hexdigest()

assert solve_ex1("a") == solve_ex1("a")
assert solve_ex1("a") != solve_ex1("b")
try:
    solve_ex1(123)
except ValueError:
    pass

print("练习1框架可运行")  # 预期输出：练习1框架可运行


## 小结

把今天的内容压缩成 3 句话：

1. 这个主题的核心变换是 $y=f(x,k)$（或 $$y=f(x,k)\bmod n$$）。
2. 正确实现需要重视：数据表示、参数合法性、测试向量与边界条件。
3. 真正理解来自：能写出最小攻防实验，并解释现象原因。


## 随堂小测（10题）
1. 这个主题的“密钥/参数”是什么？它决定了哪些行为？
2. 为什么需要模运算/置换/异或等操作？分别带来什么效果？
3. 在你的实现里，哪一步最容易写错？你如何用测试发现它？
4. 若攻击者拥有密文（或摘要），最直接的攻击方式是什么？
5. 在什么场景下“能解密”不等于“安全”？举一个例子。
6. 是否存在“重复使用随机数/nonce/IV”的风险？为什么危险？
7. 如果输入非常长，你的实现是否会变慢？瓶颈在哪？
8. 你能写出一个最小“对照实验”，让输出变化更直观吗？
9. 如果要用于真实系统，你会替换/增强哪一部分？
10. 用一句话总结：你学到的最重要概念是什么？

> 自评：能回答 7/10 且能跑通练习，就算达标。
